# Viora — Free GPU Training (Colab / Kaggle)

Train Viora's **own** video-language model for **$0** on a free **T4 (16 GB)** — plenty for the
SigLIP + Qwen-0.5B + LoRA ("pragmatic") model. Free sessions time out (4–12 h), so this notebook
**checkpoints** to Drive/output and can **resume** across sessions.

**Before running — pick a T4, not a P100:**
- **Kaggle:** right panel → *Session options* → Accelerator → **GPU T4 x2** (30 free GPU-hrs/week)
- **Colab:** Runtime → Change runtime type → **T4 GPU**

> ⚠️ On Kaggle, do **NOT** pick **GPU P100**. The P100 is compute capability `sm_60`, which current
> PyTorch dropped support for — it fails with *"CUDA error: no kernel image is available"*. The T4 is
> `sm_75` and fully supported. Cell 1 checks this and stops early with instructions if you're on a P100.

This is Viora's own model — no wrapper, no external answering API.

In [ ]:
# 1) Confirm the GPU is compatible with the installed torch (the #1 free-GPU gotcha).
import torch
assert torch.cuda.is_available(), "No GPU! Turn on the Accelerator (Kaggle Session options / Colab Runtime)."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | GPU: {name}  (compute capability sm_{major}{minor})")

# Modern PyTorch supports sm_70+ only. Kaggle's older 'GPU P100' is sm_60 -> incompatible.
if (major, minor) < (7, 0):
    raise SystemExit(
        f"\n>>> {name} is compute capability {major}.{minor} (sm_{major}{minor}), which current "
        "PyTorch does NOT support (needs sm_70+).\n"
        ">>> FIX (one setting): switch the accelerator to a T4 (sm_75):\n"
        "      Kaggle: right panel -> 'Session options' -> Accelerator -> 'GPU T4 x2'   (NOT 'GPU P100')\n"
        "      Colab:  Runtime -> Change runtime type -> 'T4 GPU'\n"
        ">>> Then re-run from cell 1. The T4 is free and fully supported; the P100 is not.\n"
    )

# Backstop: actually run a CUDA kernel (catches any other torch/GPU mismatch).
try:
    _ = (torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")).sum().item()
    print("CUDA kernel test: OK ✓  (torch matches this GPU)")
except Exception as e:
    raise SystemExit(
        f"torch cannot run a kernel on this GPU: {e}\n"
        ">>> Switch the accelerator to 'GPU T4 x2' (Kaggle) / 'T4 GPU' (Colab) and re-run."
    )

In [ ]:
# 2) Get the Viora code from GitHub (absolute paths; re-run pulls the latest fixes)
import os
os.environ["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of hanging on a login prompt

REPO_URL = "https://github.com/garvbahl37-gif/Viora.git"
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(BASE, "viora")

# If the repo is PRIVATE: add a GitHub token in Kaggle (Add-ons -> Secrets) as
# GITHUB_TOKEN, then uncomment the next two lines:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@github.com/garvbahl37-gif/Viora.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    rc = os.system(f"cd {REPO_DIR} && git pull --ff-only")   # already cloned -> update to latest
else:
    rc = os.system(f"git clone {REPO_URL} {REPO_DIR}")
if rc != 0:
    raise SystemExit(
        "Git failed. Fix one of these:\n"
        "  1) Make the repo PUBLIC (simplest), or set GITHUB_TOKEN above for a private repo.\n"
        "  2) Enable Internet: Kaggle right panel -> Internet -> On (phone-verify your account)."
    )
%cd {REPO_DIR}
print("working dir:", os.getcwd())

In [ ]:
# 3) Install Viora WITHOUT reinstalling torch.
#    Kaggle/Colab ship a CUDA-matched torch; letting pip pull torch from PyPI causes
#    "CUDA error: no kernel image is available for execution on the device".
#    So: install the package with --no-deps, then install only the extra deps explicitly.
import torch
print("keeping torch", torch.__version__, "| cuda", torch.cuda.is_available())
!pip install -q -e . --no-deps
!pip install -q einops omegaconf pyyaml tqdm rich av webdataset peft \
    transformers safetensors huggingface_hub
# Kaggle/Colab ship an old torchao (0.10) that breaks peft's LoRA dispatch (also handled in code).
!pip uninstall -q -y torchao 2>/dev/null || true
!python scripts/validate_environment.py

In [ ]:
# 4) Choose an output dir that SURVIVES session end (so you can resume).
#    Colab -> Google Drive;  Kaggle -> /kaggle/working (downloadable).
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/viora_runs/pragmatic'
except Exception:
    OUT = '/kaggle/working/pragmatic' if os.path.isdir('/kaggle') else 'runs/pragmatic'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

In [ ]:
# 5) Data. Synthetic shards so the whole pipeline runs end-to-end for free.
#    Swap in a real dataset (MSR-VTT) later for a useful model — see docs/PRODUCTION.md.
!python scripts/build_shards.py --synthetic 3000 --out data/shards/train-%06d.tar
import glob
_n = len(glob.glob("data/shards/train-*.tar"))          # robust to the actual shard count
SHARDS = "data/shards/train-{000000..%06d}.tar" % (_n - 1)
print(f"{_n} shards -> {SHARDS}")

In [ ]:
# 5b) REAL MSR-VTT captions -> real shards. You have the videos; this fetches the captions.
#      Uses a local *videodatainfo.json* if you added one; otherwise downloads the captions
#      from HuggingFace (friedrichor/MSR-VTT -> msrvtt_train_7k.json = the 7,010 train clips).
import glob, os
INPUT_ROOT = "/kaggle/input"   # Colab: point this at your mounted dataset directory

mp4s = glob.glob(f"{INPUT_ROOT}/**/*.mp4", recursive=True)
if not mp4s:
    print(f"No .mp4 under {INPUT_ROOT} -> keeping synthetic shards from cell 5.")
    print("Add the MSR-VTT VIDEOS dataset (right panel -> Add Data) to train for real.")
else:
    parts = os.path.abspath(mp4s[0]).split(os.sep)
    VIDEOS_DIR = (os.sep.join(parts[: parts.index("input") + 2])
                  if "input" in parts else os.path.dirname(mp4s[0]))
    print(f"{len(mp4s)} videos under {VIDEOS_DIR}")

    # captions: prefer a local videodatainfo.json; else pull from HuggingFace (public, no token).
    local = ([p for p in glob.glob(f"{INPUT_ROOT}/**/*videodatainfo*.json", recursive=True)
              if "train" in os.path.basename(p).lower()]
             or glob.glob(f"{INPUT_ROOT}/**/*videodatainfo*.json", recursive=True))
    if local:
        ANNOT = local[0]
    else:
        from huggingface_hub import hf_hub_download
        print("no local captions -> downloading friedrichor/MSR-VTT captions from HuggingFace...")
        ANNOT = hf_hub_download("friedrichor/MSR-VTT", "msrvtt_train_7k.json", repo_type="dataset")
    print("captions:", ANNOT)

    # --format auto handles both a videodatainfo object AND a list of {video_id, caption}
    !python scripts/prepare_video_dataset.py \
        --videos "{VIDEOS_DIR}" --annotations "{ANNOT}" \
        --format auto --split train \
        --out data/shards/msrvtt-train-%06d.tar --maxcount 500
    _n = len(glob.glob("data/shards/msrvtt-train-*.tar"))
    if _n:
        SHARDS = "data/shards/msrvtt-train-{000000..%06d}.tar" % (_n - 1)
        print(f"OK -> training will use REAL shards: {SHARDS}")
    else:
        print("!! 0 shards written -- caption video_ids don't match your .mp4 filenames.")
        print("   your video files look like:", [os.path.basename(p) for p in mp4s[:3]])

In [ ]:
# 6) Train: LoRA on Qwen-0.5B + frozen SigLIP + Viora's trainable bridge.
#    T4 supports fp16 (NOT bf16); small num_workers for Kaggle's limited CPUs.
#    Checkpoints to {OUT} every 200 steps -> resumable across free sessions.
#    (If you hit out-of-memory, lower training.batch_size to 2.)
#
#    STEPS — a REAL run needs many more steps than the synthetic smoke:
#      quick real check : 2000-5000   (watch the loss fall; ~1 hr on a T4)
#      full fine-tune   : 20000-50000 across several free sessions (resume each time, see below)
MAX_STEPS = 5000

!python scripts/train.py \
  --model configs/model/viora_pragmatic.yaml \
  --train configs/training/pragmatic_lora.yaml \
  --shards "{SHARDS}" \
  llm.name_or_path=Qwen/Qwen2.5-0.5B-Instruct \
  training.precision=fp16 training.batch_size=4 training.num_workers=2 \
  training.gradient_checkpointing=true \
  training.max_steps={MAX_STEPS} training.save_every=200 training.log_every=20 \
  training.output_dir={OUT}

## Resuming after a session times out

Re-run cells 1–5 (and 5b if using real data), then add `training.resume=<checkpoint>` to cell 6 — e.g.:

```
  training.resume={OUT}/step_2000.pt
```

It restores model + optimizer + step and continues. Repeat across free sessions until done —
this is how you reach 20k–50k steps on a free GPU. Old `step_*.pt` are auto-pruned (newest 3 kept)
so they don't fill Kaggle's 20 GB disk.

## Full-fledged (real) dataset — MSR-VTT

Cell **5b** turns a real dataset into Viora shards. On Kaggle: right panel → **Add Data** → search
**MSR-VTT** → add a version with the `.mp4` clips + a `*videodatainfo.json*`, then run cell 5b (it
auto-finds them). ~6.5k train clips × ~20 captions each = ~130k caption views.

**Any other dataset (MSVD, your own videos):** make a JSON sidecar `{ "clip1": "a caption",
"clip2": ["cap a", "cap b"] }` and run:

```
python scripts/prepare_video_dataset.py --videos <dir> --annotations captions.json \
    --format folder --out data/shards/train-%06d.tar
```

### What this training gives you (honest scope)

MSR-VTT is **caption** supervision, so the model learns to **describe** clips and answer
open-ended *"what is happening / what is in the video"* questions. It will **not** reliably do
counting, yes/no, or precise spatial questions — that needs real **QA** data (e.g. MSRVTT-QA), which
you'd feed through the adapter's `question`/`answer` fields. The **evidence timestamps** are only
meaningful once you also train on a **temporal-grounding** set (e.g. Charades-STA); on caption-only
data the grounding head stays untrained. Bigger LLM (`Qwen/Qwen2.5-1.5B-Instruct`) + more steps +
more data = better quality.

## Serve it

```
VIORA_MODEL_CONFIG=configs/model/viora_pragmatic.yaml VIORA_CHECKPOINT={OUT}/final.pt \
  uvicorn viora.serving.api:app --host 0.0.0.0 --port 8000
```